# how to finetune llm on instruction model datasets 

In [1]:
!pip install -q -U transformers datasets bitsandbytes trl peft huggingface_hub

In [2]:
from huggingface_hub import notebook_login

In [3]:
notebook_login()

In [4]:
## download model

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

config_8bit = BitsAndBytesConfig(load_in_8bit=True)

model_8bit = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config= config_8bit,
    trust_remote_code =True,
    device_map = "auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [5]:
from datasets import load_dataset

dataset = load_dataset("HumanLLMs/Human-Like-DPO-Dataset", split="train[:300]")

README.md:   0%|          | 0.00/1.94k [00:00<?, ?B/s]

data.json: reconstructing file:   0%|          |  0.00B / 28.0MB            

data.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10884 [00:00<?, ? examples/s]

In [6]:
dataset

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 300
})

In [7]:
dataset[0]

{'prompt': 'Oh, I just saw the best meme - have you seen it?',
 'chosen': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣",
 'rejected': "I'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?"}

In [8]:
example_text = [{"role": "assistant", "content": dataset[0]["chosen"]}]
example_text

[{'role': 'assistant',
  'content': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣"}]

In [9]:
tokenizer.apply_chat_template(example_text, tokenize = False)


"<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant\n😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣<|im_end|>\n"

### Note
- we can see that the system message appers even though we dont need it

In [10]:
def apply_chat_temp(example):
    example["prompt"] = [{"role": "user", "content": example["prompt"]}]

    return example


new_dataset = dataset.map(apply_chat_temp)

new_dataset[0]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

{'prompt': [{'role': 'user',
   'content': 'Oh, I just saw the best meme - have you seen it?'}],
 'chosen': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣",
 'rejected': "I'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?"}

In [11]:
new_template = """<|im_start|>assistant\n{model_answer}<|im_end|>\n"""

In [12]:
def format_dataset(example):
    example["prompt"] = tokenizer.apply_chat_template(
        example["prompt"], tokenize=False
    )
    example["chosen"] = new_template.format(model_answer=example["chosen"])
    example["rejected"] = new_template.format(model_answer=example["rejected"])
    return example
formatted_dataset = new_dataset.map(format_dataset)
formatted_dataset[0]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

{'prompt': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nOh, I just saw the best meme - have you seen it?<|im_end|>\n',
 'chosen': "<|im_start|>assistant\n😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣<|im_end|>\n",
 'rejected': "<|im_start|>assistant\nI'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?<|im_end|>\n"}

In [13]:
model_8bit

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear8bitLt(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear8bitLt(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear8bitLt(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear8bitLt(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear8bitLt(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear8bitLt(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear8bitLt(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Q

In [14]:
from peft import LoraConfig , get_peft_model

lora_config = LoraConfig(
    r=32,
    lora_alpha= 32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout = 0.05,
    bias="none",
    task_type="CAUSAL_LM" 
    
)
lora_model = get_peft_model(model_8bit, lora_config)

lora_model.print_trainable_parameters()
 



trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364


In [15]:
lora_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): 

In [18]:
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    output_dir = "./results",
    per_device_train_batch_size = 40,
    ## in reality give epoch
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    optim = "adamw_torch",
    warmup_steps = 5,
    logging_steps = 10,
    ## it uses a beta model as the reference and the value normally tells how determinent the output on the base model
    beta = 0.1,
    report_to = "none"
)

In [22]:
trainer = DPOTrainer(
    model = lora_model,
    # ref_model => u can point to a specific model fine tuned one , if left unset , it will use the non lora version of the current model
    train_dataset = formatted_dataset,
    processing_class = tokenizer,
    args=dpo_config
)

Adding EOS to train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

In [24]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=".*inputs will be cast from torch.bfloat16 to float16 during quantization.*",
)
trainer.train()

OutOfMemoryError: CUDA out of memory. Tried to allocate 236.00 MiB. GPU 0 has a total capacity of 15.58 GiB of which 30.25 MiB is free. Including non-PyTorch memory, this process has 15.54 GiB memory in use. Of the allocated memory 14.82 GiB is allocated by PyTorch, and 614.07 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)